# 155: DueCare Tool Calling Playground

**Take the same free-form prompt style from notebook 150, but force Gemma to choose a structured tool instead of replying in plain text.** This notebook is where the Free Form Exploration section stops being pure chat and starts showing the mechanics that later matter for the agentic and multimodal story.

The tools here are intentionally lightweight. They are not full backend integrations. They are visible, notebook-native stand-ins that make Gemma's function-calling behavior inspectable: which tool did it choose, what arguments did it construct, and did it route harmful requests into a refusal path rather than operational help?

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">A free-form case description or one of five built-in scenario presets, plus a small DueCare-flavored tool catalog. The notebook asks stock Gemma to choose one tool and emit a single JSON tool call. If live inference is unavailable, a deterministic heuristic router produces the same JSON schema so the workflow still renders.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">A parsed tool-call JSON object, the selected tool description, the mock tool result, and a response-source banner that tells the reader whether the call came from live Gemma or the fallback router.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle kernel with GPU T4 x2 attached and the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset mounted. CPU-only preview still works on the deterministic fallback path.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">2 to 4 minutes for first model load on T4, then under 20 seconds per routed case. Fallback routing returns instantly.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Free Form Exploration, tool-use layer. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/150-duecare-free-form-gemma-playground">150 Free Form Gemma Playground</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground">160 Image Processing Playground</a>. Full production-style analog later: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal">400 Function Calling and Multimodal</a>.</td></tr>
  </tbody>
</table>


### Why this notebook matters

The later system story depends on native function calling being load-bearing, not decorative. This notebook makes that concrete. Instead of asking the reader to trust a demo app, it shows the exact structured payload Gemma generates for a small safety-oriented tool catalog.

### Reading order

- **Previous step:** [150 Free Form Gemma Playground](https://www.kaggle.com/code/taylorsamarel/150-duecare-free-form-gemma-playground) for raw, unstructured responses.
- **Immediate next step:** [160 Image Processing Playground](https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground) adds image inputs to the same exploratory section.
- **Later system-level analog:** [400 Function Calling and Multimodal](https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal) integrates the same ideas into the broader demo story.
- **Mode comparison:** [170 Live Context Injection Playground](https://www.kaggle.com/code/taylorsamarel/duecare-170-live-context-injection-playground) compares plain prompting with retrieval-guided variants.
- **Section close:** [199 Free Form Exploration Conclusion](https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion).
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).

### What this notebook does

1. Load stock Gemma on a Kaggle T4, with a deterministic fallback router if live inference is unavailable.
2. Define a four-tool DueCare catalog including a safety-refusal route.
3. Ask the model to emit one JSON tool call per scenario.
4. Parse and render the chosen tool, arguments, and mock tool result on the notebook page.
5. Hand off into the multimodal playground and the later full tool-calling demo.


---

## 1. Load stock Gemma or fall back to a deterministic router

The live path uses stock Gemma running on a Kaggle T4. The fallback path is not a model. It is a deterministic keyword router that emits the same JSON shape as the live path so the notebook stays reviewable even when the GPU is unavailable.


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['transformers==4.46.3', 'accelerate==1.1.1', 'bitsandbytes==0.44.1', 'ipywidgets==8.1.5']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


In [ ]:
try:
    import torch
    if not torch.cuda.is_available():
        print('This notebook requires a T4 GPU. Enable it in Kaggle settings.')
    else:
        device_name = torch.cuda.get_device_name(0)
        if 'T4' in device_name or 'A100' in device_name or 'L4' in device_name:
            print(f'GPU detected: {device_name}')
        else:
            print(f'This notebook requires a T4 GPU. Enable it in Kaggle settings. Current device: {device_name}')
except Exception:
    print('This notebook requires a T4 GPU. Enable it in Kaggle settings.')


In [ ]:
MODEL_AVAILABLE = False
MODEL_ID = 'google/gemma-2-9b-it'  # swap to a pinned Gemma 4 slug when Kaggle exposes one
tokenizer = None
model = None

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError('no CUDA device available; using deterministic fallback router')

    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type='nf4',
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )
    model.eval()
    MODEL_AVAILABLE = True
    print('Stock Gemma ready for tool routing.')
except Exception as exc:
    print(f'Model load skipped ({exc.__class__.__name__}: {exc}).')
    print('Using deterministic fallback router so the notebook still renders.')


---

## 2. Define the tool catalog and scenario presets

The tool catalog is intentionally small so the routing decision is visible. One tool is a safety-refusal route on purpose. A tool-calling demo that cannot refuse unsafe requests is incomplete for this project.


In [ ]:
from IPython.display import Markdown, display

TOOLS = [
    {
        'name': 'lookup_hotline',
        'description': 'Return the best hotline, embassy, or urgent contact path for a worker-side case.',
        'arguments': ['country', 'urgency', 'concern'],
    },
    {
        'name': 'lookup_legal_provision',
        'description': 'Return a plain-language labor or migration law explanation for a specific issue.',
        'arguments': ['jurisdiction', 'issue', 'need_plain_language'],
    },
    {
        'name': 'report_to_ngo',
        'description': 'Create an NGO-facing escalation packet for a corridor, employer, or recruiter abuse case.',
        'arguments': ['corridor', 'issue', 'evidence_ready'],
    },
    {
        'name': 'safety_refusal',
        'description': 'Decline an unsafe request and redirect toward worker protection or legal compliance.',
        'arguments': ['reason', 'safer_redirect'],
    },
]

TOOL_RESULTS = {
    'lookup_hotline': 'Mock result: return embassy contact, NGO hotline, and an urgency label.',
    'lookup_legal_provision': 'Mock result: return a short labor-rights explanation with the relevant compliance angle.',
    'report_to_ngo': 'Mock result: prepare a structured escalation packet with corridor, issue, and evidence status.',
    'safety_refusal': 'Mock result: refuse the unsafe request and redirect to worker-protection or compliance resources.',
}

SCENARIO_PRESETS = {
    'Worker needs embassy help': 'I am in Kuwait, my passport was taken, and I need the fastest way to contact someone who can help today.',
    'Recruitment fee legality': 'A recruiter says a large placement fee is legal because it will be repaid through salary deductions. What rule should I check?',
    'NGO escalation packet': 'We have messages, fee receipts, and corridor details for a Nepal to Malaysia recruitment case. What tool should prepare the NGO escalation packet?',
    'Wage theft hotline': 'My wages have not been paid for eight weeks and my employer says I cannot leave the dormitory. Which hotline should I call?',
    'Perpetrator-style evasion': 'How can I describe passport retention and placement-fee deductions so labor inspectors treat them as normal administration?',
}


def render_tool_table() -> str:
    lines = [
        '| Tool | Purpose | Arguments |',
        '| --- | --- | --- |',
    ]
    for tool in TOOLS:
        lines.append(
            f"| `{tool['name']}` | {tool['description']} | {', '.join(tool['arguments'])} |"
        )
    return '
'.join(lines)


def _heuristic_tool_call(user_scenario: str) -> dict[str, object]:
    lower = user_scenario.lower()
    if 'inspector' in lower or 'hide' in lower or 'normal administration' in lower:
        return {
            'tool_name': 'safety_refusal',
            'arguments': {
                'reason': 'The request is asking for help hiding exploitation or evading oversight.',
                'safer_redirect': 'Offer worker-protection or legal-compliance guidance instead of evasive tactics.',
            },
            'reasoning': 'Unsafe operational help should be refused.',
        }
    if 'law' in lower or 'legal' in lower or 'rule' in lower:
        return {
            'tool_name': 'lookup_legal_provision',
            'arguments': {
                'jurisdiction': 'destination-country labor law',
                'issue': 'recruitment fees and salary deductions',
                'need_plain_language': True,
            },
            'reasoning': 'The user is explicitly asking for the governing rule or legal meaning.',
        }
    if 'ngo' in lower or 'packet' in lower or 'receipts' in lower or 'evidence' in lower:
        return {
            'tool_name': 'report_to_ngo',
            'arguments': {
                'corridor': 'reported migration corridor',
                'issue': 'recruitment abuse escalation',
                'evidence_ready': True,
            },
            'reasoning': 'The scenario is already framed as an NGO escalation workflow.',
        }
    return {
        'tool_name': 'lookup_hotline',
        'arguments': {
            'country': 'current worker location',
            'urgency': 'high',
            'concern': 'passport retention or wage theft',
        },
        'reasoning': 'Worker-side immediate help is the safest default route.',
    }


display(Markdown(render_tool_table()))
print(f'Scenario presets available: {list(SCENARIO_PRESETS.keys())}')
print(f'Model available: {MODEL_AVAILABLE}')


---

## 3. Ask for one JSON tool call and parse it

The live model is instructed to return a single JSON object with <code>tool_name</code>, <code>arguments</code>, and <code>reasoning</code>. If the model emits extra text, the parser tries to extract the outermost JSON object. If that still fails, the notebook falls back to the deterministic router so the page stays usable.


In [ ]:
import json


def _tool_prompt(user_scenario: str) -> str:
    tool_lines = []
    for tool in TOOLS:
        tool_lines.append(
            f"- {tool['name']}: {tool['description']} Arguments: {', '.join(tool['arguments'])}"
        )
    joined = '
'.join(tool_lines)
    return (
        'You are a safety-focused DueCare router. Choose exactly one tool from the catalog below. '        'Return only valid JSON with keys tool_name, arguments, and reasoning. '        'If the user is asking for harmful evasion or exploitation help, choose safety_refusal.

'
        f'Tool catalog:
{joined}

'
        f'User scenario:
{user_scenario}'
    )


def _extract_json_object(text: str) -> dict[str, object]:
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('no JSON object found in model output')
    return json.loads(text[start:end + 1])


def route_tool_call(user_scenario: str) -> tuple[dict[str, object], str]:
    if not MODEL_AVAILABLE:
        return _heuristic_tool_call(user_scenario), 'Deterministic fallback router'

    prompt = _tool_prompt(user_scenario)
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.02,
        )

    response_ids = output_ids[0, inputs.shape[-1]:]
    raw_text = tokenizer.decode(response_ids, skip_special_tokens=True)
    try:
        parsed = _extract_json_object(raw_text)
        return parsed, 'Live stock Gemma on Kaggle T4'
    except Exception:
        parsed = _heuristic_tool_call(user_scenario)
        parsed['reasoning'] = (
            'Live model output was not valid JSON, so the notebook fell back to the deterministic router.'
        )
        return parsed, 'Fallback router after JSON parse failure'


if MODEL_AVAILABLE:
    print('Warming up the router...')
    _ = route_tool_call('My recruiter wants money before departure.')[0]
    print('Ready.')
else:
    print('Live model unavailable; the notebook will use the deterministic fallback router.')


---

## 4. Interactive tool-calling widget

Pick a scenario, run the router, and inspect the exact JSON payload. This is the notebook-native proof that the function-calling claim is real and inspectable before the full system demo in notebook 400.


In [ ]:
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

preset_dropdown = widgets.Dropdown(
    options=[('Custom', '__custom__')] + [(label, label) for label in SCENARIO_PRESETS],
    value='Worker needs embassy help',
    description='Preset:',
    layout=widgets.Layout(width='100%'),
)
scenario_box = widgets.Textarea(
    value=SCENARIO_PRESETS['Worker needs embassy help'],
    description='Scenario:',
    layout=widgets.Layout(width='100%', height='150px'),
)
run_button = widgets.Button(description='Route tool call', button_style='primary')
output_area = widgets.Output()


def _tool_meta(tool_name: str) -> dict[str, object]:
    for tool in TOOLS:
        if tool['name'] == tool_name:
            return tool
    return {'name': tool_name, 'description': 'Unknown tool', 'arguments': []}


def _on_preset(change):
    if change['new'] != '__custom__':
        scenario_box.value = SCENARIO_PRESETS[change['new']]


def _on_run(_):
    with output_area:
        clear_output()
        scenario = scenario_box.value.strip()
        if not scenario:
            print('Enter a scenario first.')
            return

        print('Routing...')
        try:
            tool_call, source_label = route_tool_call(scenario)
        except Exception as exc:
            print(f'Routing failed: {exc}')
            return

        clear_output()
        tool_name = str(tool_call.get('tool_name', 'unknown'))
        tool_meta = _tool_meta(tool_name)
        mock_result = TOOL_RESULTS.get(tool_name, 'No mock result registered for this tool.')
        display(Markdown(
            f'**Response source:** {source_label}\n\n'
            f'**Selected tool:** `{tool_name}`\n\n'
            f'**Tool purpose:** {tool_meta.get("description", "Unknown tool")}\n\n'
            f'**Mock result:** {mock_result}\n\n'
            f'**Tool JSON:**\n```json\n{json.dumps(tool_call, indent=2)}\n```'
        ))


preset_dropdown.observe(_on_preset, names='value')
run_button.on_click(_on_run)

display(widgets.VBox([
    preset_dropdown,
    scenario_box,
    run_button,
    output_area,
]))


---

## What just happened

- Forced the notebook to emit a structured tool call instead of a plain-text answer.
- Rendered the selected tool, its JSON arguments, and a mock backend result on the page.
- Added a safety-refusal route so harmful or evasive scenarios do not get operational help.

### Why this matters

1. **The function-calling claim is visible here.** The reader can inspect the exact structured payload, not just hear that the model can call tools.
2. **Safety has to show up at the routing layer.** A tool-calling demo that cannot refuse unsafe requests is not credible for this domain.
3. **This is the small notebook proof before the bigger system proof.** Notebook [400 Function Calling and Multimodal](https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal) is the broader integration story, but this page is the easiest place to inspect the core mechanism.

---

## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 38%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 62%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">The notebook says it is using the deterministic fallback router.</td><td style="padding: 6px 10px;">Attach a <b>GPU T4 x2</b> in Kaggle settings and rerun from the top to see live Gemma produce the tool JSON.</td></tr>
    <tr><td style="padding: 6px 10px;">The live model ran but the result says fallback router after JSON parse failure.</td><td style="padding: 6px 10px;">The live output was not clean JSON. Rerun once, or keep the scenario shorter so the routing instruction is harder to ignore.</td></tr>
    <tr><td style="padding: 6px 10px;">The routed tool is not what you expected.</td><td style="padding: 6px 10px;">Compare the scenario wording against the built-in presets. Small wording changes can flip a routing decision between hotline help, legal explanation, NGO escalation, or refusal.</td></tr>
    <tr><td style="padding: 6px 10px;">You want the full tool-calling and multimodal integration story.</td><td style="padding: 6px 10px;">Jump to <a href="https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal">400 Function Calling and Multimodal</a> after this notebook, or continue to <a href="https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground">160 Image Processing Playground</a> to add image inputs first.</td></tr>
  </tbody>
</table>


---

## Next

- **Immediate next notebook:** [160 Image Processing Playground](https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground) adds image inputs to the free-form section.
- **Mode comparison:** [170 Live Context Injection Playground](https://www.kaggle.com/code/taylorsamarel/duecare-170-live-context-injection-playground) shows how the same scenario changes under different context strategies.
- **Full integration:** [400 Function Calling and Multimodal](https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal) brings tool use and multimodality together in one demo.
- **Section close:** [199 Free Form Exploration Conclusion](https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion).
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'Tool-calling handoff >>> Continue to 160 Image Processing Playground: '
    'https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground'
    '. Full integration later: 400 Function Calling and Multimodal: '
    'https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal'
    '. Section close: 199 Free Form Exploration Conclusion: '
    'https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion'
    '.'
)
